# AIC211 · Topic 2 · Tuần 4 — TV1 (Trí): Dữ liệu + Baseline đầu tiên (bản Kaggle)

**Mục tiêu tuần này:** có dữ liệu sạch đã chia theo domain (`A.parquet`, `B.parquet`) và baseline LR-F1 cho ra số trên A-test và B.
Đây là tuần nặng nhất của TV1. Nhựt (TV2) đang chờ `A.parquet` để test T1–T7 trên URL thật.

| Bước | Làm gì | Ước tính |
|---|---|---|
| 0 | Tạo Drive chung, GitHub, tải 2 dataset (ngoài Colab) | 45 phút |
| 1–2 | Cài thư viện, ghi `src/common.py` | 5 phút |
| 3–6 | Nạp nguồn A, B → làm sạch → chia theo domain → lưu | 30 phút |
| 7 | EDA (bằng chứng artifact) | 30 phút |
| 8–9 | Baseline LR-F1, thử F2 | 20 phút |

Chạy **từ trên xuống, từng ô một** (Shift + Enter). Ô nào lỗi thì dừng lại, đọc dòng lỗi cuối cùng.

## Bước 0 — Làm ngoài Colab (làm 1 lần)

**0.1. Drive chung.** Vào Google Drive → tạo thư mục `AIC211_URL` ngay trong *My Drive* → chuột phải → *Share* → thêm Gmail của Nhựt, quyền **Editor**.

**0.2. GitHub.** Tạo repo private `AIC211_URL`, mời Nhựt làm collaborator. Repo chỉ chứa code (`src/`, notebook), **không** đưa dữ liệu lên.

**0.3. Tải nguồn A — Malicious URLs dataset (Kaggle).** https://www.kaggle.com/datasets/sid321axn/malicious-urls-dataset → đăng nhập → *Download* → giải nén → upload `malicious_phish.csv` lên `AIC211_URL/data/raw/`.
(Không dùng ISCX-URL2016 nữa: các bản tải được chỉ có file feature `All.csv`, `Phishing.csv`..., **không có chuỗi URL**.)

**0.3b. Xóa kết quả cũ (nếu từng chạy bản ISCX).** Trong Drive `AIC211_URL`, xóa hết file trong `data/processed/`, `models/`, `results/`. Mấy file ISCX trong `data/raw/` để đó hay xóa đều được.

**0.4. Tải nguồn B — PhiUSIIL.** https://archive.ics.uci.edu/dataset/967/phiusiil+phishing+url+dataset → *Download* → giải nén → upload `PhiUSIIL_Phishing_URL_Dataset.csv` lên `AIC211_URL/data/raw/`.

> An toàn: chỉ xử lý chuỗi URL như văn bản. **Không** bấm, không mở bất kỳ URL nào trong dataset.

In [ ]:
# Ô 1 — Cài thư viện + nối Google Drive (chạy lại mỗi lần mở Colab)
import os, sys, importlib
try:
    import google.colab  # chỉ có khi chạy trên Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install tldextract rapidfuzz shap xgboost
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/AIC211_URL'   # thư mục chung của nhóm
else:
    ROOT = os.path.abspath('AIC211_URL')          # chạy thử trên máy (không phải Colab)

for d in ['data/raw', 'data/processed', 'src', 'models', 'results', 'figures', 'notebooks']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)
sys.path.insert(0, f'{ROOT}/src')
print('ROOT =', ROOT)
print(sorted(os.listdir(ROOT)))

## Bước 2 — File chung `src/common.py`
Chứa mọi hàm dùng chung: chuẩn hóa URL, chia domain, feature F1/F2, ngưỡng, metric, ghi kết quả. Nhựt sẽ `import common` từ file này, nên **muốn sửa hàm nào thì báo Nhựt trước**.

In [ ]:
# Ô 2 — Ghi file src/common.py (file "hợp đồng" TV1 sở hữu, TV2 sẽ import)
COMMON_CODE = r"""
import re, math, os, csv, datetime
from collections import Counter
from urllib.parse import unquote
import numpy as np, pandas as pd
import tldextract
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import confusion_matrix, average_precision_score, f1_score

EXTRACT = tldextract.TLDExtract(suffix_list_urls=())   # danh sách đuôi domain đóng gói sẵn, không cần mạng

def reg_domain_of_host(host):
    ext = EXTRACT(host)
    return f"{ext.domain}.{ext.suffix}" if ext.suffix else ext.domain   # IP -> chính IP đó

def normalize_url(u):
    # Mọi nguồn về CÙNG một dạng: bỏ scheme, bỏ 'www.', host chữ thường, giải mã %xx 1 lần, bỏ #fragment
    u = unquote(str(u).strip())
    u = re.sub(r'^[a-zA-Z][a-zA-Z0-9+.-]*://', '', u)
    u = u.split('#', 1)[0]
    host = re.match(r'[^/?]*', u).group(0)
    rest = u[len(host):]
    host = host.lower()
    if host.startswith('www.'):
        host = host[4:]
    return host + rest

def split_parts(url_norm):
    host = re.match(r'[^/?]*', url_norm).group(0)
    return host, url_norm[len(host):]

def prepare_source(df, url_col='url', label_col='label'):
    # df: cột url (chuỗi gốc) + label (0 = benign, 1 = malicious). Trả về df sạch + log số dòng sau mỗi bước.
    log = {'raw': len(df)}
    df = df[[url_col, label_col]].dropna().rename(columns={url_col: 'url', label_col: 'label'})
    df['url_norm'] = df['url'].map(normalize_url)
    df = df[df['url_norm'].str.len() > 3]
    log['after_dropna'] = len(df)
    n_lab = df.groupby('url_norm')['label'].transform('nunique')
    df = df[n_lab == 1]                                   # URL mang cả 2 nhãn -> bỏ
    log['after_drop_conflict'] = len(df)
    df = df.drop_duplicates('url_norm')
    log['after_exact_dedup'] = len(df)
    parts = df['url_norm'].map(split_parts)
    df['host'] = parts.str[0]
    df['rest'] = parts.str[1]
    df['reg_domain'] = df['host'].map(reg_domain_of_host)
    return df.reset_index(drop=True), log

def domain_split(df, seed=42):
    # 70/15/15 theo registered domain (20 fold: 3 test, 3 val, 14 train)
    sgkf = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=seed)
    fold = np.empty(len(df), dtype=int)
    for i, (_, idx) in enumerate(sgkf.split(df, df['label'], df['reg_domain'])):
        fold[idx] = i
    df = df.copy()
    df['split'] = np.where(fold < 3, 'test', np.where(fold < 6, 'val', 'train'))
    assert df.groupby('reg_domain')['split'].nunique().max() == 1, 'RÒ RỈ: có domain nằm ở 2 tập'
    return df

def remove_cross_source_overlap(df_b, df_a_train):
    # Nguồn B: bỏ URL trùng nguyên văn và URL có domain đã nằm trong train của A
    log = {'B_before': len(df_b)}
    df_b = df_b[~df_b['url_norm'].isin(set(df_a_train['url_norm']))]
    log['B_after_exact'] = len(df_b)
    df_b = df_b[~df_b['reg_domain'].isin(set(df_a_train['reg_domain']))]
    log['B_after_domain'] = len(df_b)
    df_b = df_b.copy(); df_b['split'] = 'test'
    return df_b.reset_index(drop=True), log

SUSPICIOUS = ['login', 'signin', 'verify', 'account', 'update', 'secure', 'bank', 'confirm',
              'password', 'webscr', 'billing', 'suspend', 'unlock', 'wallet', 'free', 'bonus']
IP_RE = re.compile(r'^\d{1,3}(\.\d{1,3}){3}(:\d+)?$')

def entropy(s):
    if not s: return 0.0
    c = Counter(s); n = len(s)
    return -sum(v / n * math.log2(v / n) for v in c.values())

def f1_features(url_norm):
    host, rest = split_parts(url_norm)
    path, _, query = rest.partition('?')
    low = url_norm.lower()
    return {
        'len_url': len(url_norm), 'len_host': len(host), 'len_path': len(path), 'len_query': len(query),
        'n_dots': url_norm.count('.'), 'n_dots_host': host.count('.'),
        'n_hyphen': url_norm.count('-'), 'n_hyphen_host': host.count('-'),
        'n_digits': sum(ch.isdigit() for ch in url_norm),
        'n_digits_host': sum(ch.isdigit() for ch in host),
        'n_special': sum(not ch.isalnum() for ch in url_norm),
        'n_slash': path.count('/'), 'n_params': query.count('&') + (1 if query else 0),
        'has_ip': int(bool(IP_RE.match(host))), 'has_at': int('@' in url_norm),
        'has_punycode': int('xn--' in host),
        'n_susp_kw': sum(kw in low for kw in SUSPICIOUS),
        'entropy_url': entropy(url_norm), 'entropy_host': entropy(host),
    }

def make_F1(df):
    return pd.DataFrame([f1_features(u) for u in df['url_norm']], index=df.index)

def make_F2_vectorizer():
    from sklearn.compose import ColumnTransformer
    from sklearn.feature_extraction.text import TfidfVectorizer
    tf = lambda: TfidfVectorizer(analyzer='char', ngram_range=(3, 5), min_df=3,
                                 max_features=200_000, sublinear_tf=True)
    return ColumnTransformer([('host', tf(), 'host'), ('path', tf(), 'rest')])

def get_scores(model, X):
    # Điểm độc hại liên tục (càng cao càng độc)
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    return model.decision_function(X)

def threshold_at_fpr(y_val, s_val, target_fpr=0.01):
    # Chọn ngưỡng trên VALIDATION nguồn A sao cho FPR ≈ 1%. Chốt xong KHÔNG đổi nữa.
    return float(np.quantile(np.asarray(s_val)[np.asarray(y_val) == 0], 1 - target_fpr))

def evaluate(y, s, thr):
    y = np.asarray(y); pred = (np.asarray(s) > thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {'precision': tp / max(tp + fp, 1), 'recall': tp / max(tp + fn, 1),
            'f1': f1_score(y, pred, zero_division=0), 'fpr': fp / max(fp + tn, 1),
            'pr_auc': average_precision_score(y, s), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}

def log_result(root, exp, model, source, metrics, defense='none', seed=0, k=0, ops='-', mode='-'):
    # Mọi kết quả đều ghi qua hàm này vào results/results.csv
    path = f'{root}/results/results.csv'
    row = {'time': datetime.datetime.now().isoformat(timespec='seconds'), 'exp': exp, 'model': model,
           'defense': defense, 'source': source, 'seed': seed, 'k': k, 'ops': ops, 'mode': mode, **metrics}
    new = not os.path.exists(path)
    with open(path, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(row));
        if new: w.writeheader()
        w.writerow(row)
"""
with open(f'{ROOT}/src/common.py', 'w', encoding='utf-8') as f:
    f.write(COMMON_CODE)
import common; importlib.reload(common)
from common import *
print('Đã ghi', f'{ROOT}/src/common.py')
print(normalize_url('HTTPS://WWW.Example.com/Login%20Page#top'))

## Bước 3–5 — Nạp dữ liệu
Ô 3 kiểm tra đã có đủ 2 file dữ liệu chưa. Ô 4 có dòng tự kiểm tra: nếu URL là tên giả kiểu `iscx-url-123.com` thì báo lỗi và dừng.

In [ ]:
# Ô 3 — Kiểm tra đã có 2 file dữ liệu trong data/raw chưa
import glob
need = {'Nguồn A (Kaggle)': 'malicious_phish.csv', 'Nguồn B (PhiUSIIL)': 'PhiUSIIL*.csv'}
for name, pat in need.items():
    hits = glob.glob(f'{ROOT}/data/raw/**/{pat}', recursive=True)
    print(f'{name:20s}:', os.path.relpath(hits[0], ROOT) if hits else 'CHƯA CÓ — xem lại Bước 0')

In [ ]:
# Ô 4 — Nạp nguồn A (Malicious URLs, Kaggle): benign = 0; phishing + malware = 1; bỏ defacement
p = glob.glob(f'{ROOT}/data/raw/**/malicious_phish.csv', recursive=True)
assert p, 'Chưa thấy malicious_phish.csv trong data/raw/ — xem Bước 0.3'
raw = pd.read_csv(p[0])
print('Các cột:', list(raw.columns))
print(raw['type'].value_counts())

raw = raw[raw['type'].isin(['benign', 'phishing', 'malware'])]
src_A = pd.DataFrame({'url': raw['url'].astype(str),
                      'label': (raw['type'] != 'benign').astype(int)})

# Lấy mẫu 100.000 URL cho Colab chạy nhanh (ghi vào báo cáo). Muốn dùng toàn bộ thì xóa dòng này.
src_A = src_A.sample(n=min(100_000, len(src_A)), random_state=42).reset_index(drop=True)

# Tự kiểm tra: URL phải là URL thật, không phải tên giả kiểu iscx-url-123.com
fake = src_A['url'].str.fullmatch(r'iscx-url-\d+\.com').mean()
assert fake < 0.01, f'{fake:.0%} URL là tên giả — kiểm tra lại file dữ liệu!'

print(src_A['label'].value_counts().rename({0: 'benign', 1: 'malicious'}))
print(src_A.sample(5, random_state=0).to_string())   # PHẢI thấy URL thật

In [ ]:
# Ô 5 — Nạp nguồn B (PhiUSIIL). Chỉ lấy cột URL + label, bỏ hết feature có sẵn.
phi_path = glob.glob(f'{ROOT}/data/raw/**/PhiUSIIL*.csv', recursive=True)
assert phi_path, 'Chưa thấy PhiUSIIL_Phishing_URL_Dataset.csv trong data/raw/'
phi = pd.read_csv(phi_path[0], usecols=['URL', 'label'])

# KIỂM TRA quy ước nhãn trước khi đảo: trang UCI ghi 1 = legitimate, 0 = phishing.
# Nhìn mắt: các URL nhãn 1 phải là trang quen (google, wikipedia...).
for lab in [1, 0]:
    print(f'--- label = {lab} ---')
    print(phi[phi['label'] == lab]['URL'].head(5).str[:70].to_string())

phi['label'] = 1 - phi['label']      # đảo lại cho thống nhất: 1 = malicious
src_B = phi.rename(columns={'URL': 'url'})
print(src_B['label'].value_counts().rename({0: 'benign', 1: 'malicious'}))

## Bước 6 — Làm sạch + chia theo domain
Dòng `assert` sẽ báo lỗi ngay nếu có domain nằm ở 2 tập. Bảng log là số liệu bắt buộc trong báo cáo (*số URL bị loại ở mỗi bước*).

In [ ]:
# Ô 6 — Làm sạch, chia theo domain, lưu parquet + log số URL bị loại
A, log_A = prepare_source(src_A)
A = domain_split(A, seed=42)
B, log_B0 = prepare_source(src_B)
B, log_B1 = remove_cross_source_overlap(B, A[A.split == 'train'])

A.to_parquet(f'{ROOT}/data/processed/A.parquet')
B.to_parquet(f'{ROOT}/data/processed/B.parquet')

log_table = pd.DataFrame([{'nguồn': 'A', **log_A}, {'nguồn': 'B', **log_B0, **log_B1}])
log_table.to_csv(f'{ROOT}/results/dedup_log.csv', index=False)
display(log_table)                                   # bảng này đưa thẳng vào báo cáo
display(A.groupby(['split', 'label']).size().unstack())
print('B:', B['label'].value_counts().to_dict())

## Bước 7 — EDA
Điều cần nhìn: tỉ lệ có `http://`/`www.` có khác hẳn giữa 2 lớp hoặc 2 nguồn không; benign có toàn là trang chủ (không có path) không. Khác nhiều = artifact, ghi lại cho E1. Hình lưu ở `figures/`.

In [ ]:
# Ô 7 — EDA trên dữ liệu GỐC (bằng chứng artifact cho E1)
import matplotlib.pyplot as plt

def raw_stats(src, name):
    u = src['url'].astype(str)
    d = pd.DataFrame({'label': src['label'].map({0: 'benign', 1: 'malicious'}),
                      'has_scheme': u.str.contains(r'^[a-zA-Z]+://', regex=True),
                      'has_www': u.str.lower().str.replace(r'^[a-z]+://', '', regex=True).str.startswith('www.')})
    return d.groupby('label')[['has_scheme', 'has_www']].mean().assign(source=name)

display(pd.concat([raw_stats(src_A, 'A'), raw_stats(src_B, 'B')]).round(3))

both = pd.concat([A.assign(source='A'), B.assign(source='B')])
both['len_url'] = both['url_norm'].str.len()
both['has_path'] = both['rest'].str.len() > 1
both['tld'] = both['host'].map(lambda h: EXTRACT(h).suffix or '(ip)')
display(both.groupby(['source', 'label'])[['has_path', 'len_url']].agg(['mean']).round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, src in zip(axes, ['A', 'B']):
    for lab, name in [(0, 'benign'), (1, 'malicious')]:
        x = both[(both.source == src) & (both.label == lab)]['len_url'].clip(upper=200)
        ax.hist(x, bins=50, alpha=.6, label=name, density=True)
    ax.set_title(f'Độ dài URL — nguồn {src}'); ax.set_xlabel('số ký tự (cắt ở 200)'); ax.legend()
plt.tight_layout(); plt.savefig(f'{ROOT}/figures/eda_url_length.png', dpi=200); plt.show()

top_tld = both.groupby(['source', 'label'])['tld'].agg(lambda s: s.value_counts().head(5).to_dict())
display(top_tld.to_frame('top 5 TLD'))

## Bước 8–9 — Baseline
Ngưỡng chọn trên **validation của A** sao cho chỉ ~1% URL sạch bị chặn nhầm (FPR ≈ 1%). Tuyệt đối không dùng B để chỉnh gì. Kết quả tự ghi vào `results/results.csv`.

In [ ]:
# Ô 8 — Baseline đầu tiên: Logistic Regression trên F1 (LR-F1)
import joblib, json
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

FA, FB = make_F1(A), make_F1(B)
tr, va, te = (A.split == 'train'), (A.split == 'val'), (A.split == 'test')

lr_f1 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced'))
lr_f1.fit(FA[tr], A.label[tr])

thr = threshold_at_fpr(A.label[va], get_scores(lr_f1, FA[va]))    # chọn trên VAL của A, không nhìn B
rows = []
for name, X, y in [('A-test', FA[te], A.label[te]), ('B', FB, B.label)]:
    m = evaluate(y, get_scores(lr_f1, X), thr)
    log_result(ROOT, 'baseline_w4', 'LR_F1', name, m)
    rows.append({'tập': name, **m})
display(pd.DataFrame(rows).round(3))

joblib.dump(lr_f1, f'{ROOT}/models/LR_F1.joblib')
print('Ngưỡng tạm (chưa đóng băng, tuần 5 mới chốt):', round(thr, 4))

In [ ]:
# Ô 9 — (Kiểm tra nhanh) F2 char n-gram chạy được không, trên 20% train cho nhanh
lr_f2 = make_pipeline(make_F2_vectorizer(), LogisticRegression(max_iter=2000, C=4, class_weight='balanced'))
small = A[tr].sample(frac=0.2, random_state=0)
lr_f2.fit(small[['host', 'rest']], small['label'])
thr2 = threshold_at_fpr(A.label[va], get_scores(lr_f2, A[va][['host', 'rest']]))
print('LR-F2 (20% train) trên A-test:', {k: round(float(v), 3) for k, v in
      evaluate(A.label[te], get_scores(lr_f2, A[te][['host', 'rest']]), thr2).items() if k in ('recall', 'fpr', 'f1')})

## Xong tuần 4 khi
- [ ] `data/processed/A.parquet`, `B.parquet` có đủ cột: url, url_norm, host, rest, reg_domain, label, split
- [ ] `results/dedup_log.csv` có số URL bị loại từng bước
- [ ] Hình EDA trong `figures/`; ghi 3–5 dòng nhận xét artifact vào file báo cáo chung
- [ ] LR-F1 có số trên A-test và B (`results/results.csv`)
- [ ] **Nhắn Nhựt:** "A.parquet xong rồi" để Nhựt chạy lại notebook TV2 trên URL thật

Tuần 5 (để sau): train RF, XGB, LR-F2, SVM-F2 → chốt `thresholds.json` → E1 (SHAP) → E2.